# kbprojection functionality walkthrough

This notebook explores the main public pieces of [`ettoc00/kbprojection`](https://github.com/ettoc00/kbprojection): data models, prompt construction, LLM output parsing, KB filtering/post-processing, dataset loader APIs, LangPro parsing helpers, and the high-level orchestration entry points.

The first cells are designed to run without OpenAI/Anthropic/Gemini/LangPro credentials. Optional cells near the end show where live LLM, dataset download, and LangPro calls fit into the workflow.


## 1. Install or import the repository

This cell imports `kbprojection` if it is already installed. If not, it clones the GitHub repository into a local `kbprojection_repo/` folder beside this notebook and installs it in editable mode.


In [6]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/ettoc00/kbprojection"
REPO_DIR = Path.cwd() / "kbprojection_repo"

if importlib.util.find_spec("kbprojection") is None:
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])

import kbprojection
print("kbprojection version:", getattr(kbprojection, "__version__", "unknown"))
print("package loaded from:", kbprojection.__file__)


kbprojection version: unknown
package loaded from: None


## 2. Create core NLI objects

The package normalizes NLI examples into `NLIProblem` instances and uses Pydantic configuration/result models throughout the pipeline.


In [7]:
from kbprojection.models import NLIProblem, NLILabel, ProblemConfig, TestMode, ExperimentResult

problem = NLIProblem(
    id="manual-001",
    premises=["A little girl in pink boots runs down the street."],
    hypothesis="A human is running outdoors.",
    gold_label=NLILabel.ENTAILMENT,
    dataset="manual",
    split="demo",
)

config = ProblemConfig(
    llm_provider="openai",
    model="gpt-4o",
    prompt_style="icl",
    test_mode=TestMode.BOTH,
    run_ablation=False,
    verbose=True,
)

empty_result = ExperimentResult(problem=problem)
print(problem.model_dump())
print(config.model_dump())
print(empty_result.final_status)


{'id': 'manual-001', 'premises': ['A little girl in pink boots runs down the street.'], 'hypothesis': 'A human is running outdoors.', 'gold_label': <NLILabel.ENTAILMENT: 'entailment'>, 'dataset': 'manual', 'split': 'demo', 'original_data': None}
{'llm_provider': 'openai', 'model': 'gpt-4o', 'prompt_style': 'icl', 'post_process': True, 'test_mode': <TestMode.BOTH: 'both'>, 'run_ablation': False, 'verbose': True}
ExperimentStatus.UNKNOWN


## 3. Inspect and fill prompt templates

`kbprojection.prompts` stores legacy and newer prompt templates. `fill_prompt` inserts the premise and hypothesis for a specific problem.


In [9]:
from kbprojection.prompts import list_prompts, get_prompt, fill_prompt

print("Available prompts:", list_prompts())
print("\nICL prompt preview:")
print(get_prompt("icl")[:700], "...")

filled_prompt = fill_prompt("icl", problem.premises, problem.hypothesis)
print("\nFilled prompt tail:")
print(filled_prompt[-500:])


Available prompts: ['legacy_cot', 'legacy_least_to_most', 'legacy_icl', 'icl', 'cot']

ICL prompt preview:
You are a knowledge base injection assistant. Your task is to generate semantic relations that bridge the meaning gap between a Premise and Hypothesis.

## STRICT RULES (Must follow exactly)

1. **Output format**: Your output MUST be enclosed in delimiters:
   [KB_START]
   predicate(arg1, arg2)
   [KB_END]

2. **Allowed predicates only**:
   - `isa_wn(X, Y)` = X is a type/kind of Y (hypernymy). Example: isa_wn(dog, animal)
   - `disj(X, Y)` = X and Y are mutually exclusive (cannot both be true). Example: disj(sit, stand)

3. **ALWAYS use base forms (lemmas)**:
   - CORRECT: isa_wn(run, move)
   - WRONG: isa_wn(running, moving)
   - CORRECT: disj(walk, ride)
   - WRONG: disj(walking, riding) ...

Filled prompt tail:
 man is drawing a gun.
[KB_START]
isa_wn(aim, draw)
[KB_END]
Note: "aim" and "draw" are NOT opposites - aiming can follow drawing. Use isa_wn if one action implies the

## 4. Parse LLM output into KB injections

The newer prompts ask an LLM to place relations between `[KB_START]` and `[KB_END]`. `extract_kb_from_output` extracts valid `isa_wn(...)` and `disj(...)` lines from either delimited or legacy-style output.


In [10]:
from kbprojection.llm import extract_kb_from_output

mock_llm_output = """
Reasoning omitted here.
[KB_START]
isa_wn(girl, human)
isa_wn(street, outdoors)
disj(sit, stand)
not_allowed(foo, bar)
[KB_END]
"""

kb_raw = extract_kb_from_output(mock_llm_output)
print(kb_raw)


['isa_wn(girl, human)', 'isa_wn(street, outdoors)', 'disj(sit, stand)']


## 5. Parse, normalize, and generate KB candidate variants

The filtering module handles low-level KB syntax and derives variants such as lemmatized and swapped relations. These functions are useful for debugging generated KB before a LangPro call.


In [11]:
from kbprojection.filtering import (
    parse_kb_injection,
    normalize_kb_args,
    remove_underscores,
    generate_all_candidates,
)

examples = [
    "isa_wn(little_girl, human)",
    "disj(sitting, standing)",
]

for item in examples:
    print("raw:", item)
    print("parsed:", parse_kb_injection(item))
    print("normalized:", normalize_kb_args(item))
    print()

candidates = generate_all_candidates(
    pred="isa_wn",
    arg1="little_girl",
    arg2="human",
    original_text="isa_wn(little_girl, human)",
    post_process=True,
)

for candidate in candidates:
    print(f"{candidate.relation:35s} provenance={candidate.provenance}")


raw: isa_wn(little_girl, human)
parsed: ('isa_wn', 'little_girl', 'human')
normalized: isa_wn(little girl, human)

raw: disj(sitting, standing)
parsed: ('disj', 'sitting', 'standing')
normalized: disj(sitting, standing)



[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


isa_wn(little girl, human)          provenance=post_process
isa_wn(human, little girl)          provenance=post_process_swap


## 6. Run the KB filtering pipeline

`pipeline_filter_kb_injections` keeps only allowed predicates and relations whose arguments can be found in the premise/hypothesis. It may download small NLTK resources the first time it runs.


In [ ]:
from kbprojection.filtering import pipeline_filter_kb_injections

kb_candidates = [
    "isa_wn(girl, human)",
    "isa_wn(street, outdoors)",
    "isa_wn(camera, device)",       # camera/device are not in this example
    "causes(run, tired)",           # unsupported predicate
    "this is malformed",            # invalid syntax
]

filtered = pipeline_filter_kb_injections(
    kb_candidates,
    premise=problem.premises,
    hypothesis=problem.hypothesis,
    post_process=True,
    use_semantic=False,
)

for result in filtered:
    print(f"kept: {result.relation:30s} provenance={result.provenance} original={result.original_text}")


## 7. Use the dataset loader API with a tiny local dataset

The repository provides SNLI and SICK loaders. To avoid a large network download in this walkthrough, this cell creates a minimal loader subclass that exercises the same base-class methods: `load`, `iter_problems`, `get_problem`, and `random_problem`.


In [ ]:
import json
from pathlib import Path
from typing import Iterator, List, Optional

from kbprojection.loaders.base import DatasetLoader
from kbprojection.models import NLIProblem
from kbprojection.utils import get_smallest_problems

class MiniNLILoader(DatasetLoader):
    SPLITS = ["dev"]

    def _download(self) -> None:
        path = self._get_file_path("dev")
        if path.exists():
            return
        rows = [
            {
                "id": "mini-1",
                "premises": ["A dog is running in a park."],
                "hypothesis": "An animal is moving outdoors.",
                "gold_label": "entailment",
            },
            {
                "id": "mini-2",
                "premises": ["A woman is smiling."],
                "hypothesis": "A woman is frowning.",
                "gold_label": "contradiction",
            },
        ]
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")

    def _get_splits(self) -> List[str]:
        return self.SPLITS

    def _get_file_path(self, split: str) -> Path:
        return self.data_dir / f"mini_{split}.jsonl"

    def _iter_file(self, file_path: Path) -> Iterator[dict]:
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)

    def _parse_row(self, row: dict, split: str) -> Optional[NLIProblem]:
        return NLIProblem(
            id=row["id"],
            premises=row["premises"],
            hypothesis=row["hypothesis"],
            gold_label=self._parse_label(row["gold_label"]),
            dataset="mini",
            split=split,
            original_data=row,
        )

mini_loader = MiniNLILoader(data_dir=Path.cwd() / "demo_data")
mini_loader.load(splits=["dev"])

print("All problems:")
for prob in mini_loader.iter_problems("dev"):
    print(prob.id, prob.gold_label.value, "::", prob.premises[0], "=>", prob.hypothesis)

print("get_problem:", mini_loader.get_problem("mini-1", split="dev"))
print("random entailment:", mini_loader.random_problem("dev", label_filter={"entailment"}).id)
print("smallest entailments:", get_smallest_problems(mini_loader, split="dev", limit=5, label_filter={"entailment"}))


## 8. Parse LangPro-style structures

`kbprojection.langpro` contains helper classes and parsers for Prolog-like terms, category/type structures, and KB entries. These are independent of the live LangPro API.


In [ ]:
from kbprojection.langpro import Atom, Integer, Var, Compound, parse_kb, parse_caty

print("Atoms and compounds:")
print(Atom("dog"))
print(Integer(3))
print(Var("X"))
print(Compound("isa_wn", ["dog", "animal"]))

kb_as_dicts = [
    {"functor": "isa_wn", "args": ["girl", "human"]},
    {"functor": "disj", "args": ["sit", "stand"]},
]
print("\nParsed KB:")
for rel in parse_kb(kb_as_dicts):
    print(repr(rel), "=>", str(rel))

print("\nParsed category:", parse_caty({"functor": "/", "args": ["S", "NP"]}))


## 9. Optional: load a real SICK/SNLI example

The built-in `SICKLoader` and `SNLILoader` automatically download data if files are missing. Run this cell only when network access is available.


In [ ]:
# Optional network-dependent example.
# from kbprojection import SICKLoader, SNLILoader
#
# sick = SICKLoader(data_dir=Path.cwd() / "data" / "sick")
# sick.load(splits=["dev"])
# first_sick = next(sick.iter_problems("dev"))
# print(first_sick)
#
# snli = SNLILoader(data_dir=Path.cwd() / "data" / "snli")
# snli.load(splits=["dev"])
# print(snli.random_problem("dev", label_filter={"entailment"}))


## 10. Optional: call a live LLM

This requires one of the provider API keys described by the repository (`OPENAI_API_KEY`, `OPENROUTER_API_KEY`, `GEMINI_API_KEY`, or `ANTHROPIC_API_KEY`). The output feeds directly into the filtering pipeline.


In [ ]:
# Optional API-dependent example.
# from kbprojection.llm import call_llm
#
# generated_kb = call_llm(
#     provider="openai",
#     model="gpt-4o",
#     prompt_style="icl",
#     prob=problem,
# )
# print("Generated KB:", generated_kb)
#
# filtered_generated_kb = pipeline_filter_kb_injections(
#     generated_kb,
#     premise=problem.premises,
#     hypothesis=problem.hypothesis,
# )
# print("Filtered KB:", [str(x) for x in filtered_generated_kb])


## 11. Optional: process a problem through the orchestration pipeline

`process_single_problem` runs the full sequence: baseline LangPro call, LLM KB generation, KB filtering, and LangPro re-checks. This requires the LangPro API endpoint to be reachable and an LLM provider key to be configured.


In [ ]:
# Optional API-dependent example.
# from kbprojection.orchestration import process_single_problem
#
# live_config = ProblemConfig(
#     llm_provider="openai",
#     model="gpt-4o",
#     prompt_style="icl",
#     test_mode=TestMode.BOTH,
#     run_ablation=False,
#     verbose=True,
# )
#
# result = process_single_problem(problem, config=live_config)
# print(result.model_dump_json(indent=2))
